In [1]:
import tensorflow as tf
import tensorflow_hub as hub
import numpy as np
import cv2
print("TF version: ", tf.__version__)

TF version:  2.20.0


In [4]:
model = hub.load("https://tfhub.dev/google/movenet/singlepose/lightning/4")

movenet = model.signatures['serving_default']

In [5]:
KEYPOINT_NAMES = [
    'nose',
    'left_eye',
    'right_eye',
    'left_ear',
    'right_ear',
    'left_shoulder',
    'right_shoulder',
    'left_elbow',
    'right_elbow',
    'left_wrist',
    'right_wrist',
    'left_hip',
    'right_hip',
    'left_knee',
    'right_knee',
    'left_ankle',
    'right_ankle'
]

NOSE = 0
L_SHOULDER, R_SHOULDER = 5, 6
L_ELBOW,    R_ELBOW    = 7, 8
L_WRIST,    R_WRIST    = 9, 10
L_HIP,      R_HIP      = 11, 12
L_KNEE,     R_KNEE     = 13, 14
L_ANKLE,    R_ANKLE    = 15, 16



SKELETON = [
    (NOSE, L_SHOULDER),  (NOSE, R_SHOULDER),
    (L_SHOULDER, R_SHOULDER),
    (L_SHOULDER, L_ELBOW), (L_ELBOW, L_WRIST),
    (R_SHOULDER, R_ELBOW), (R_ELBOW, R_WRIST),
    (L_SHOULDER, L_HIP),  (R_SHOULDER, R_HIP),
    (L_HIP, R_HIP),
    (L_HIP, L_KNEE),   (L_KNEE, L_ANKLE),
    (R_HIP, R_KNEE),   (R_KNEE, R_ANKLE),
]

In [7]:
def preprocess_frame(frame):

  img = tf.convert_to_tensor(frame)

  img = tf.expand_dims(img, axis=0)

  img = tf.image.resize_with_pad(img, target_height=192, target_width=192)

  img = tf.cast(img, dtype=tf.int32)

  return img

In [8]:
def run_movenet(input_tensor):
  outputs = movenet(input = input_tensor)

  keypoints_raw = outputs['output_0']

  keypoints = keypoints_raw.numpy()[0, 0]

  return keypoints